<a href="https://colab.research.google.com/github/matveyburets-droid/-/blob/main/DZ_3_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Задача 1
import numpy as np
import requests
import urllib.request
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
class Data:
    def __init__(self, data):
        self.data = sorted(list(data))
        self.n = len(self.data)

    def count(self):
        return self.n

    def sum(self):
        return sum(self.data)

    def moment(self, k):
        if self.n == 0:
            return 0.0
        return sum(x ** k for x in self.data) / self.n

    def mean(self):
        return self.moment(1)

    def var(self):
        m = self.mean()
        return sum((x - m) ** 2 for x in self.data) / self.n

    def std(self):
        return self.var() ** 0.5

    def skewness(self):
        m = self.mean()
        s = self.std()
        if s == 0:
            return 0.0
        third_moment_central = sum((x - m) ** 3 for x in self.data) / self.n
        return third_moment_central / (s ** 3)

    def kurtosis(self):
        m = self.mean()
        s = self.std()
        if s == 0:
            return 0.0
        fourth_moment_central = sum((x - m) ** 4 for x in self.data) / self.n
        return fourth_moment_central / (s ** 4)

    def percentile(self, p):
        if not self.data:
            return None
        if p <= 0:
            return float(self.data[0])
        if p >= 1:
            return float(self.data[-1])
        pos = p * (self.n - 1)
        idx = int(pos)
        rest = pos - idx
        if idx + 1 < self.n:
            return float(self.data[idx] + rest * (self.data[idx + 1] - self.data[idx]))
        else:
            return float(self.data[idx])

    def median(self):
        return self.percentile(0.5)

    def __str__(self):
        return (
            f"Count: {self.count()}\n"
            f"Mean: {self.mean():.6f}\n"
            f"Variance: {self.var():.6f}\n"
            f"Standard deviation: {self.std():.6f}\n"
            f"Skewness: {self.skewness():.6f}\n"
            f"Kurtosis: {self.kurtosis():.6f}\n"
            f"2nd moment: {self.moment(2):.6f}\n"
            f"3rd moment: {self.moment(3):.6f}\n"
            f"4th moment: {self.moment(4):.6f}\n"
            f"Median: {self.median():.6f}\n"
            f"25% quantile: {self.percentile(0.25):.6f}\n"
            f"75% quantile: {self.percentile(0.75):.6f}"
        )
url = (
    "https://www.cbr.ru/currency_base/dynamics/"
    "?UniDbQuery.Posted=True"
    "&UniDbQuery.so=1"
    "&UniDbQuery.mode=1"
    "&UniDbQuery.date_req1="
    "&UniDbQuery.date_req2="
    "&UniDbQuery.VAL_NM_RQ=R01235"
    "&UniDbQuery.From=10.01.2025"
    "&UniDbQuery.To=31.12.2025"
)

response = requests.get(url)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

rates = []

for row in soup.find_all("tr"):
    cells = row.find_all(["td", "th"])

    if len(cells) >= 3:
        date_text = cells[0].get_text(strip=True)
        rate_text = cells[2].get_text(strip=True)

        try:
            date = datetime.strptime(date_text, "%d.%m.%Y")


            rate = float(rate_text.replace(",", "."))

            rates.append((date, rate))

        except ValueError:

            continue



if not rates:
    raise RuntimeError("Не удалось найти курсы валют на странице ЦБ РФ.")


# ЦБ выводит даты от новых к старым,
# поэтому сортируем от старых к новым
rates.sort(key=lambda x: x[0])

# Считаем доходности
returns = []
returns = []

for i in range(1, len(rates)):
    previous_rate = rates[i - 1][1]
    current_rate = rates[i][1]

    ret = (current_rate / previous_rate - 1) * 100
    returns.append(ret)

# Финальный ответ
data = Data(returns)
print(data)

Count: 246
Mean: -0.104923
Variance: 0.808571
Standard deviation: 0.899206
Skewness: 0.030392
Kurtosis: 4.833100
2nd moment: 0.819580
3rd moment: -0.233570
4th moment: 3.204075
Median: -0.095644
25% quantile: -0.481985
75% quantile: 0.345359


In [5]:
# Для Евро (код R01239).

url = (
    "https://www.cbr.ru/currency_base/dynamics/"
    "?UniDbQuery.Posted=True"
    "&UniDbQuery.so=1"
    "&UniDbQuery.mode=1"
    "&UniDbQuery.date_req1="
    "&UniDbQuery.date_req2="
    "&UniDbQuery.VAL_NM_RQ=R01239"
    "&UniDbQuery.From=10.01.2025"
    "&UniDbQuery.To=31.12.2025"
)

response = requests.get(url)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
rates = []

for row in soup.find_all("tr"):
    cells = row.find_all(["td", "th"])

    if len(cells) >= 3:
        date_text = cells[0].get_text(strip=True)
        rate_text = cells[2].get_text(strip=True)

        try:
            date = datetime.strptime(date_text, "%d.%m.%Y")

            # В русской версии ЦБ десятичный разделитель — запятая
            rate = float(rate_text.replace(",", "."))

            rates.append((date, rate))

        except ValueError:

            continue



if not rates:
    raise RuntimeError("Не удалось найти курсы валют на странице ЦБ РФ.")



rates.sort(key=lambda x: x[0])

# Считаем доходности
returns = []
returns = []

for i in range(1, len(rates)):
    previous_rate = rates[i - 1][1]
    current_rate = rates[i][1]

    ret = (current_rate / previous_rate - 1) * 100
    returns.append(ret)

# Финальный ответ
data = Data(returns)
print(data)

Count: 246
Mean: -0.048386
Variance: 1.051010
Standard deviation: 1.025188
Skewness: -0.019848
Kurtosis: 4.608529
2nd moment: 1.053351
3rd moment: -0.174061
4th moment: 5.109591
Median: -0.045471
25% quantile: -0.551865
75% quantile: 0.480209


In [6]:
# Задача 2
def palindrome(s: str) -> bool:
  # Очистка от знаков препинания и пробелов
  for ch in " .,!?-—;:\"'()[]{}":
    s = s.replace(ch, "")
  s = s.lower()
  return s == s[::-1]
print(palindrome("Карл у Клары украл кораллы."))
print(palindrome("И любит Сева вестибюли!"))

False
True


In [14]:
# Задача 3
import urllib.request

url = "https://raw.githubusercontent.com/matveyburets-droid/-/refs/heads/main/words-list-russian.txt"

urllib.request.urlretrieve(url, "words-list-russian.txt")
from collections import Counter

def typesetter(filename, source_word):
    source_letters = Counter(source_word.lower())
    count = 0

    with open(filename, "r", encoding="utf-8") as file:
        for line in file:
            word = line.strip().lower()

            if not word:
                continue

            if Counter(word) <= source_letters:
                count += 1

                if count <= 10:
                    print(word)

    print("Количество слов:", count)


typesetter("words-list-russian.txt", "автоинспектор")

автоинспектор
автор
автоспорт
аист
аконит
акр
аксон
анкер
анортит
ант
Количество слов: 565


In [15]:
# Задача 4
import urllib.request


url = "https://raw.githubusercontent.com/matveyburets-droid/-/refs/heads/main/words-list-russian.txt"
urllib.request.urlretrieve(url, "words-list-russian.txt")


def wordChain(filename, s1, s2):

    # Слова должны быть одинаковой длины
    if len(s1) != len(s2):
        return []

    # Загружаем допустимые слова
    with open(filename, "r", encoding="utf-8") as file:
        words = {line.strip().lower() for line in file if line.strip()}

    s1 = s1.lower()
    s2 = s2.lower()

    # Оба слова должны быть в словаре
    if s1 not in words or s2 not in words:
        return []

    # Уже одинаковые слова
    if s1 == s2:
        return [s1]


    queue = [s1]

    # Для каждого найденного слова запоминаем предыдущее
    previous = {s1: None}

    while queue:
        current = queue.pop(0)

        # Перебираем все возможные замены одной буквы
        for i in range(len(current)):
            for letter in "абвгдеёжзийклмнопрстуфхцчшщъыьэюя":

                if letter == current[i]:
                    continue

                new_word = current[:i] + letter + current[i + 1:]

                if new_word in words and new_word not in previous:
                    previous[new_word] = current


                    if new_word == s2:
                        chain = []
                        word = s2

                        while word is not None:
                            chain.append(word)
                            word = previous[word]

                        chain.reverse()
                        return chain

                    queue.append(new_word)

    return []


print(wordChain(
    "words-list-russian.txt",
    "муха",
    "слон"
))

['муха', 'мура', 'тура', 'тара', 'кара', 'каре', 'кафе', 'кафр', 'каюр', 'каюк', 'каик', 'клик', 'клин', 'клон', 'слон']
